# STAGE 13B.3 — Matched Pseudo-Absence Control Recovery

**Malaysia-only landslide FYP | Conservative candidate screening and extraction manifest**

**Goal:** investigate the 33 Stage 13B.2 control-only failures using **unused static-valid background candidates**. This notebook **does not** fabricate temporal features, certify replacement controls, modify original datasets, or retrain models. It produces a shortlist requiring historical feature extraction and QC re-validation.

Existing 121-case temporal model and Stage 07B–12 benchmarks are retained unchanged.

# PART A — PROJECT PATHS AND SOURCE VALIDATION
## 1. Imports and local project configuration

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

AUDIT_DIR = DEV_DIR / "05_model_results" / "stage13b2_temporal_reconciliation"
STATIC_PATH = DEV_DIR / "03_derived_features" / "static_features_final" / "static_feature_samples_final.csv"
ANNOTATED_PATH = DEV_DIR / "03_derived_features" / "temporal_features_final" / "temporal_feature_samples_final_annotated.csv"
ORIGINAL_FULL_PATH = DEV_DIR / "03_derived_features" / "temporal_features_final" / "temporal_model_full_enhanced.csv"
RECOVERY_PATH = AUDIT_DIR / "case_recovery_decision_matrix.csv"
ROOT_CAUSE_PATH = AUDIT_DIR / "excluded_group_root_causes.csv"
OUTPUT_DIR = DEV_DIR / "05_model_results" / "stage13b3_control_recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create several provisional choices per required slot; candidates are not approved controls.
SHORTLIST_PER_SLOT = 5
MAX_RADIUS_KM = 250.0  # Soft shortlist radius only; never a substitute for validation.

for label, path in {
    "Static candidate inventory": STATIC_PATH,
    "Original annotated matched groups": ANNOTATED_PATH,
    "Previous recovery decision matrix": RECOVERY_PATH,
    "Previous root cause table": ROOT_CAUSE_PATH,
    "Original full temporal model": ORIGINAL_FULL_PATH,
}.items():
    print(f"{label}: {path.exists()} — {path}")
    if not path.exists():
        raise FileNotFoundError(f"Required source not found: {path}")


Static candidate inventory: True — C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final\static_feature_samples_final.csv
Original annotated matched groups: True — C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_feature_samples_final_annotated.csv
Previous recovery decision matrix: True — C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b2_temporal_reconciliation\case_recovery_decision_matrix.csv
Previous root cause table: True — C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b2_temporal_reconciliation\excluded_group_root_causes.csv
Original full temporal model: True — C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_model_full_enhanced.csv


## 2. Load sources and reject ambiguous identifiers

In [2]:
static = pd.read_csv(STATIC_PATH, low_memory=False)
master = pd.read_csv(ANNOTATED_PATH, low_memory=False)
decisions = pd.read_csv(RECOVERY_PATH, low_memory=False)
root_causes = pd.read_csv(ROOT_CAUSE_PATH, low_memory=False)
original_full = pd.read_csv(ORIGINAL_FULL_PATH, low_memory=False)

for name, data, cols in [
    ("static", static, ["sample_id", "sample_class", "national_core_static_valid_final", "longitude", "latitude"]),
    ("matched master", master, ["sample_id", "matched_positive_id", "temporal_role", "reference_date", "longitude", "latitude"]),
    ("recovery matrix", decisions, ["matched_positive_id", "candidate_status", "case_valid_full_enhanced"]),
    ("root causes", root_causes, ["matched_positive_id", "recovery_category"]),
]:
    missing = sorted(set(cols) - set(data.columns))
    if missing:
        raise KeyError(f"{name} missing required columns: {missing}")

for name, data in [("static", static), ("matched master", master)]:
    if data["sample_id"].isna().any() or data["sample_id"].duplicated().any():
        raise ValueError(f"{name}: sample_id must be complete and unique")

if decisions["matched_positive_id"].duplicated().any():
    raise ValueError("Decision matrix has duplicate matched_positive_id values")

print("Static rows:", len(static), "| Annotated matched rows:", len(master))
print("Stage13B.2 decisions:", decisions["candidate_status"].value_counts(dropna=False).to_dict())


Static rows: 3200 | Annotated matched rows: 567
Stage13B.2 decisions: {'ALREADY_INCLUDED': 121, 'CASE_AND_OR_CONTROL_QC_FAILURE': 35, 'CASE_VALID_CONTROL_FAILURE': 33}


# PART B — VALID GROUPS AND INVALID CONTROL SLOTS
## 3. Recover exactly the Stage 13B.2 control-only group IDs

In [3]:
control_only = root_causes.loc[
    root_causes["recovery_category"].eq("CONTROL_ONLY_FAILURE"),
    "matched_positive_id"
].dropna().astype(str).unique().tolist()

confirm = decisions.set_index("matched_positive_id")
for group_id in control_only:
    if group_id not in confirm.index:
        raise ValueError(f"Control-only group missing in decision matrix: {group_id}")
    if not bool(confirm.loc[group_id, "case_valid_full_enhanced"]):
        raise ValueError(f"Control-only group contains an invalid case: {group_id}")

print("Groups with valid cases / invalid controls:", len(control_only))
if len(control_only) != 33:
    print("NOTE: Different from earlier 33-group audit. Verify which source version is being used.")

# Flags are taken from the saved Stage 05B-QC output, not inferred from missingness alone.
needed_flags = [
    "rainfall_valid", "soil_moisture_complete", "landcover_valid_strict", "waterway_available"
]
missing_flags = [name for name in needed_flags if name not in master.columns]
if missing_flags:
    raise KeyError("Annotated master missing original QC flags: " + str(missing_flags))

master["full_row_valid_saved"] = master[needed_flags].fillna(False).astype(bool).all(axis=1)

slots = []
for group_id in control_only:
    rows = master.loc[master["matched_positive_id"].astype(str).eq(group_id)].copy()
    cases = rows.loc[rows["temporal_role"].eq("case")]
    controls = rows.loc[rows["temporal_role"].eq("matched_control")]
    if len(cases) != 1 or len(controls) != 2:
        raise ValueError(f"Unexpected matched-group structure: {group_id}")
    if not bool(cases["full_row_valid_saved"].iloc[0]):
        raise ValueError(f"Valid-case premise violated: {group_id}")
    for _, ctrl in controls.loc[~controls["full_row_valid_saved"]].iterrows():
        slots.append({
            "matched_positive_id": group_id,
            "old_control_sample_id": str(ctrl["sample_id"]),
            "reference_date": str(cases["reference_date"].iloc[0]),
            "case_longitude": pd.to_numeric(cases["longitude"].iloc[0], errors="coerce"),
            "case_latitude": pd.to_numeric(cases["latitude"].iloc[0], errors="coerce"),
            "failed_qc_flags": "; ".join([f for f in needed_flags if not bool(ctrl[f])]),
            "old_control_longitude": pd.to_numeric(ctrl["longitude"], errors="coerce"),
            "old_control_latitude": pd.to_numeric(ctrl["latitude"], errors="coerce"),
        })

slot_df = pd.DataFrame(slots)
slot_df.to_csv(OUTPUT_DIR / "control_replacement_slots.csv", index=False)
print("Control replacement slots required:", len(slot_df))
print("Affected unique historical cases:", slot_df["matched_positive_id"].nunique())
display(slot_df.head(10))


Groups with valid cases / invalid controls: 33
Control replacement slots required: 34
Affected unique historical cases: 33


,matched_positive_id,old_control_sample_id,reference_date,case_longitude,case_latitude,failed_qc_flags,old_control_longitude,old_control_latitude
0,POS_0008,BG_01201,2011-11-25 00:00:00,101.151900,4.312000,soil_moisture_complete,109.851962,1.647907
1,POS_0013,BG_00761,2016-07-22 00:00:00,101.756000,3.108200,landcover_valid_strict,103.047419,1.959683
2,POS_0020,BG_02991,2014-11-27 19:00:00,102.002500,2.698500,soil_moisture_complete,118.954802,5.404230
3,POS_0022,BG_02981,2016-08-30 15:00:00,101.677787,3.185783,landcover_valid_strict,119.165035,5.313570
4,POS_0023,BG_01305,2013-10-23 01:00:00,101.393200,4.420200,soil_moisture_complete,111.034818,1.641848
5,POS_0024,BG_01667,2017-05-10 16:00:00,101.967410,2.718877,landcover_valid_strict,113.145445,2.030758
6,POS_0046,BG_01562,2017-11-10 00:00:00,100.430408,5.355873,landcover_valid_strict,112.794300,2.207360
7,POS_0051,BG_01255,2011-01-29 04:00:00,118.065700,5.845200,soil_moisture_complete,110.766792,1.558273
8,POS_0053,BG_01295,2017-11-20 00:00:00,101.673790,2.991997,landcover_valid_strict,111.412739,1.200832
9,POS_0057,BG_01188,2014-05-17 10:40:00,101.576800,3.221100,soil_moisture_complete,104.264868,1.423896


## 4. Determine unused background candidates without duplicate controls

In [4]:
valid_background = static.loc[
    static["sample_class"].eq("background_candidate")
    & static["national_core_static_valid_final"].eq(True)
].copy()

# Exclude every control that was previously assigned, including those in failed groups.
# This avoids silently moving temporal controls between matched groups.
used_ids = set(master["sample_id"].astype(str))
unused = valid_background.loc[~valid_background["sample_id"].astype(str).isin(used_ids)].copy()
unused["longitude"] = pd.to_numeric(unused["longitude"], errors="coerce")
unused["latitude"] = pd.to_numeric(unused["latitude"], errors="coerce")
unused = unused.loc[
    unused["longitude"].between(99, 120)
    & unused["latitude"].between(0, 8)
].copy()

print("Static-valid background pool:", len(valid_background))
print("Previously assigned controls excluded:", len(valid_background) - len(unused))
print("Unused candidates with plausible Malaysia coordinates:", len(unused))

if unused.empty:
    print("No unused candidate rows. New spatial sampling may be required, with the same original pseudo-absence safeguards.")


Static-valid background pool: 2930
Previously assigned controls excluded: 378
Unused candidates with plausible Malaysia coordinates: 2552


# PART C — PROVISIONAL BACKGROUND SHORTLIST
## 5. Spatial candidate ranking (not a recovery approval)

In [5]:
def haversine_km(lon, lat, lon0, lat0):
    """Distance in kilometres; used only to rank candidates for subsequent audit."""
    lon = np.deg2rad(np.asarray(lon, dtype=float))
    lat = np.deg2rad(np.asarray(lat, dtype=float))
    lon0, lat0 = np.deg2rad([float(lon0), float(lat0)])
    dlon = lon - lon0
    dlat = lat - lat0
    a = np.sin(dlat / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# The original Stage05B sampling was globally random without replacement.
# Distances below are investigation priorities only, NOT claimed original sampling criteria.
# Prefer untouched candidates; retain an alternative shortlist for manual source extraction.
ranked = []
for _, slot in slot_df.iterrows():
    if unused.empty:
        break
    if pd.isna(slot["case_longitude"]) or pd.isna(slot["case_latitude"]):
        continue
    distance = haversine_km(
        unused["longitude"].to_numpy(),
        unused["latitude"].to_numpy(),
        slot["case_longitude"], slot["case_latitude"]
    )
    order = np.argsort(distance)
    near = order[distance[order] <= MAX_RADIUS_KM]
    chosen = near[:SHORTLIST_PER_SLOT] if len(near) else order[:SHORTLIST_PER_SLOT]
    for rank, idx in enumerate(chosen, 1):
        candidate = unused.iloc[int(idx)]
        ranked.append({
            "matched_positive_id": slot["matched_positive_id"],
            "old_control_sample_id": slot["old_control_sample_id"],
            "reference_date": slot["reference_date"],
            "candidate_sample_id": str(candidate["sample_id"]),
            "candidate_longitude": float(candidate["longitude"]),
            "candidate_latitude": float(candidate["latitude"]),
            "distance_to_case_km": float(distance[int(idx)]),
            "candidate_rank": rank,
            "candidate_status": "UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPATIAL_QC",
        })

shortlist = pd.DataFrame(ranked)
shortlist.to_csv(OUTPUT_DIR / "provisional_control_shortlist.csv", index=False)
print("Provisional candidates:", len(shortlist))
print("Unique candidate locations:", shortlist["candidate_sample_id"].nunique() if not shortlist.empty else 0)
print("Note: the same candidate can appear on different shortlists; do not assign it twice.")
display(shortlist.head(10))


Provisional candidates: 170
Unique candidate locations: 72
Note: the same candidate can appear on different shortlists; do not assign it twice.


,matched_positive_id,old_control_sample_id,reference_date,candidate_sample_id,candidate_longitude,candidate_latitude,distance_to_case_km,candidate_rank,candidate_status
0,POS_0008,BG_01201,2011-11-25 00:00:00,BG_00263,101.178150,4.340972,4.341592,1,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
1,POS_0008,BG_01201,2011-11-25 00:00:00,BG_00233,101.193053,4.270633,6.479220,2,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
2,POS_0008,BG_01201,2011-11-25 00:00:00,BG_00265,101.212613,4.335245,7.210901,3,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
3,POS_0008,BG_01201,2011-11-25 00:00:00,BG_00269,101.051497,4.359263,12.310573,4,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
4,POS_0008,BG_01201,2011-11-25 00:00:00,BG_00267,101.157916,4.433945,13.576062,5,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
5,POS_0013,BG_00761,2016-07-22 00:00:00,BG_00360,101.735466,2.983178,14.087608,1,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
6,POS_0013,BG_00761,2016-07-22 00:00:00,BG_00366,101.891537,3.039185,16.893096,2,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
7,POS_0013,BG_00761,2016-07-22 00:00:00,BG_00391,101.914930,3.187754,19.738718,3,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
8,POS_0013,BG_00761,2016-07-22 00:00:00,BG_00361,101.809819,2.901678,23.729150,4,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...
9,POS_0013,BG_00761,2016-07-22 00:00:00,BG_00381,101.910860,3.255709,23.762106,5,UNVALIDATED_NEEDS_TEMPORAL_EXTRACTION_AND_SPAT...


## 6. Prepare extraction manifest for Stage 05B environmental observations

In [6]:
# Reference dates are inherited ONLY from the verified case matching information.
# Stage 05B must extract historical rainfall, ERA5 daily soil moisture,
# and historical land cover for each candidate/date pair.
manifest = shortlist.copy()
if not manifest.empty:
    manifest = manifest.rename(columns={
        "candidate_sample_id": "sample_id",
        "candidate_longitude": "longitude",
        "candidate_latitude": "latitude",
    })
    manifest["temporal_role"] = "matched_control_candidate"
    manifest["reference_date"] = pd.to_datetime(manifest["reference_date"], errors="coerce")
    manifest["reference_year"] = manifest["reference_date"].dt.year
    manifest["reference_month"] = manifest["reference_date"].dt.month
    manifest["reference_day"] = manifest["reference_date"].dt.day
    manifest["extract_rainfall"] = True
    manifest["extract_era5_soil_moisture"] = True
    manifest["extract_historical_landcover"] = True
    manifest["verify_waterway_distance"] = True
    manifest["verify_original_pseudoabsence_spatial_exclusion"] = True
    manifest["verified_replacement"] = False

manifest.to_csv(OUTPUT_DIR / "candidate_temporal_extraction_manifest.csv", index=False)
print("Rows requiring original Stage05B feature extraction:", len(manifest))
print("Unique sample/date pairs:", len(manifest[["sample_id", "reference_date"]].drop_duplicates()) if not manifest.empty else 0)


Rows requiring original Stage05B feature extraction: 170
Unique sample/date pairs: 154


# PART D — RECOVERY FEASIBILITY AND SAFETY CHECKS
## 7. Diagnose why controls failed originally

In [7]:
problem_rows = master.loc[
    master["sample_id"].astype(str).isin(slot_df["old_control_sample_id"].astype(str))
].copy()
fail_rows = []
for _, row in problem_rows.iterrows():
    failed = [f for f in needed_flags if not bool(row[f])]
    fail_rows.append({
        "matched_positive_id": row["matched_positive_id"],
        "sample_id": row["sample_id"],
        "failed_qc_flags": "; ".join(failed),
        "landcover_source": row.get("landcover_source"),
        "landcover_reliability": row.get("landcover_reliability"),
        "landcover_harmonised": row.get("landcover_harmonised"),
        "reference_date": row.get("reference_date"),
        "must_reextract_or_replace": True,
    })
failed_control_report = pd.DataFrame(fail_rows)
failed_control_report.to_csv(OUTPUT_DIR / "failed_control_diagnostics.csv", index=False)
print("Rows with failing original control QC:", len(failed_control_report))
display(failed_control_report.head())


Rows with failing original control QC: 34


,matched_positive_id,sample_id,failed_qc_flags,landcover_source,landcover_reliability,landcover_harmonised,reference_date,must_reextract_or_replace
0,POS_0008,BG_01201,soil_moisture_complete,GLC_FCS30D,HIGH,FOREST,2011-11-25 00:00:00,True
1,POS_0013,BG_00761,landcover_valid_strict,NaN,MISSING,NaN,2016-07-22 00:00:00,True
2,POS_0020,BG_02991,soil_moisture_complete,GLC_FCS30D,HIGH,WETLAND,2014-11-27 19:00:00,True
3,POS_0022,BG_02981,landcover_valid_strict,NaN,MISSING,NaN,2016-08-30 15:00:00,True
4,POS_0023,BG_01305,soil_moisture_complete,GLC_FCS30D,HIGH,FOREST,2013-10-23 01:00:00,True


## 8. Verify date and control uniqueness constraints

In [8]:
checks = []
for group_id in control_only:
    group = master.loc[master["matched_positive_id"].astype(str).eq(group_id)]
    case = group.loc[group["temporal_role"].eq("case")]
    controls = group.loc[group["temporal_role"].eq("matched_control")]
    reference_dates = pd.to_datetime(group["reference_date"], errors="coerce")
    checks.append({
        "matched_positive_id": group_id,
        "one_case_two_controls": len(case) == 1 and len(controls) == 2,
        "case_qc_passes": bool(case["full_row_valid_saved"].all()),
        "group_dates_complete": bool(reference_dates.notna().all()),
        "group_one_reference_date": reference_dates.nunique() == 1,
        "two_distinct_current_controls": controls["sample_id"].nunique() == 2,
        "replacement_slots": int(slot_df["matched_positive_id"].eq(group_id).sum()),
        "shortlist_rows": int(shortlist["matched_positive_id"].eq(group_id).sum()) if not shortlist.empty else 0,
    })
checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUTPUT_DIR / "control_recovery_group_prechecks.csv", index=False)
print("All affected matched groups have valid structure:", bool(checks_df["one_case_two_controls"].all()))
print("All affected groups have consistent dates:", bool(checks_df["group_dates_complete"].all() and checks_df["group_one_reference_date"].all()))


All affected matched groups have valid structure: True
All affected groups have consistent dates: True


## 9. Report preliminary group-level recovery feasibility

In [9]:
group_report = checks_df.copy()
group_report["candidate_shortlist_available"] = (
    group_report["shortlist_rows"] >= group_report["replacement_slots"]
)
group_report["case_ready_for_recovery_review"] = (
    group_report["one_case_two_controls"]
    & group_report["case_qc_passes"]
    & group_report["group_dates_complete"]
    & group_report["group_one_reference_date"]
    & group_report["candidate_shortlist_available"]
)
group_report["actual_recovered"] = False
# Never use availability alone as evidence that environmental values or spatial exclusions pass.
group_report["recovery_decision"] = np.where(
    group_report["case_ready_for_recovery_review"],
    "CANDIDATES_FOUND_EXTRACTION_AND_QC_REQUIRED",
    "INSUFFICIENT_CANDIDATES_OR_GROUP_REVIEW"
)
group_report.to_csv(OUTPUT_DIR / "group_recovery_feasibility.csv", index=False)
print(group_report["recovery_decision"].value_counts().to_string())


recovery_decision
CANDIDATES_FOUND_EXTRACTION_AND_QC_REQUIRED    33


# PART E — SAFE NEXT-STEP GATE
## 10. Write evidence-based decision gate and dataset invariants

In [10]:
current_case_ids = set(
    original_full.loc[original_full["temporal_role"].eq("case"), "matched_positive_id"].astype(str)
)

summary = pd.DataFrame([
    {"check": "Original temporal cases preserved", "status": "PASS" if len(current_case_ids) == 121 else "REVIEW", "result": len(current_case_ids)},
    {"check": "Stage13B2 control-only failures loaded", "status": "PASS" if len(control_only) == 33 else "REVIEW", "result": len(control_only)},
    {"check": "Invalid control slots identified", "status": "PASS" if len(slot_df) > 0 else "REVIEW", "result": len(slot_df)},
    {"check": "Unused static background candidates identified", "status": "PASS" if len(unused) > 0 else "REVIEW", "result": len(unused)},
    {"check": "All proposed candidates already have verified historical features", "status": "PENDING_EXTRACTION", "result": "NO — shortlist contains coordinates and dates only"},
    {"check": "Original pseudo-absence exclusion distances verified for new controls", "status": "PENDING_SPATIAL_AUDIT", "result": "No new candidate approved"},
    {"check": "Historical environmental source timestamps checked", "status": "PENDING_SOURCE_AUDIT", "result": "Must verify observations precede/reference event"},
    {"check": "Safe to replace production temporal dataset", "status": "NOT_YET", "result": "Zero approved replacements"},
    {"check": "Old benchmark results preserved", "status": "PASS", "result": "Original datasets and models untouched"},
])
summary.to_csv(OUTPUT_DIR / "stage13b3_decision_gate.csv", index=False)

overview = {
    "original_full_temporal_cases": len(current_case_ids),
    "control_only_groups_targeted": len(control_only),
    "replacement_slots_required": len(slot_df),
    "available_unused_background_candidates": len(unused),
    "candidate_shortlist_rows": len(shortlist),
    "groups_with_provisional_candidates": int(group_report["candidate_shortlist_available"].sum()),
    "verified_recovered_groups": 0,
    "dataset_retraining_authorized": False,
}
(OUTPUT_DIR / "stage13b3_summary.json").write_text(json.dumps(overview, indent=2), encoding="utf-8")

print(summary.to_string(index=False))
print("\nOUTPUT DIRECTORY:", OUTPUT_DIR)


                                                                check                status                                             result
                                    Original temporal cases preserved                  PASS                                                121
                               Stage13B2 control-only failures loaded                  PASS                                                 33
                                     Invalid control slots identified                  PASS                                                 34
                       Unused static background candidates identified                  PASS                                               2552
    All proposed candidates already have verified historical features    PENDING_EXTRACTION NO — shortlist contains coordinates and dates only
Original pseudo-absence exclusion distances verified for new controls PENDING_SPATIAL_AUDIT                          No new candidate approved

## Interpretation and next action

1. **Shortlisted ≠ recovered.** The original 05B temporal extraction was run for 189 cases and their matched controls, **not** for arbitrary unused background points. No replacement control should be accepted until it receives source-date-appropriate rainfall, soil-moisture and high-reliability land-cover observations, as well as current waterway context.
2. **Reproduce the original spatial pseudo-absence safeguards** for each proposed new control using the Stage 04/05A sampling code; this notebook does not establish the distance buffer or exclusion polygon on its own.
3. **Keep the already-valid control** in each group where possible; request one replacement for each invalid control slot. A candidate may appear in multiple shortlists and must not be used in more than one matched group.
4. **Preserve group structure:** exactly one positive + two unique controls, same case event reference date, reliable geospatial position, original land-cover reliability rule, no water controls, rainfall window validity, soil moisture presence, waterway distances.
5. **Do not use model scores to select controls.** All replacement decisions must be determined independently of the held-out validation labels and results. Re-run LR/RF/XGBoost and full nested spatial CV if the data are revised.

### Outputs to send back

- `stage13b3_decision_gate.csv`
- `stage13b3_summary.json`
- `control_replacement_slots.csv`
- `failed_control_diagnostics.csv`
- `provisional_control_shortlist.csv`
- `candidate_temporal_extraction_manifest.csv`
- `group_recovery_feasibility.csv`

**No production dataset will be created in this phase.** The follow-on recovery phase will run temporal extraction on the shortlist and then re-check complete matched groups.